In [3]:
import pandas as pd
import glob
import os

In [4]:
folder_path = "."
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

In [12]:
dfs = {}

for file in csv_files:
    # Extract the integer prefix from filename
    base = os.path.basename(file)
    int_name = int(base.split("_")[0])  # Convert to int
    
    # Read CSV and remove the first row
    df = pd.read_csv(file)
    df = df.iloc[1:].reset_index(drop=True)
    
    # Store in dict
    dfs[int_name] = df

In [13]:
len(dfs.keys())

12

In [25]:
# Step 1: Concatenate all dataframes into one
combined_df = pd.concat(dfs.values(), ignore_index=True)
print(f"Initial combined number of references: {combined_df.shape[0]}")

# Step 2: Remove duplicates based on PMID
before = combined_df.shape[0]
combined_df = combined_df.drop_duplicates(subset=["PMID"], keep="first")
after = combined_df.shape[0]
print(f"Removed {before - after} duplicate PMIDs. New number of references: {combined_df.shape[0]}")

# Step 3: Remove rows where PMID equals one of the int_names
int_names = set(dfs.keys())  # the integer prefixes from file names
before = combined_df.shape[0]
mask = combined_df["PMID"].isin(int_names)
rows_to_drop = combined_df[mask]
print(f"Removed {len(rows_to_drop)} rows where PMID matches int_name. This corresponds to rows that are one of the TRIPOD statements.")
combined_df = combined_df[~mask]
after = combined_df.shape[0]
print(f"Final number of references: {combined_df.shape[0]}")

# Reset index for cleanliness
combined_df = combined_df.reset_index(drop=True)

Initial combined number of references: 6762
Removed 453 duplicate PMIDs. New number of references: 6309
Removed 3 rows where PMID matches int_name. This corresponds to rows that are one of the TRIPOD statements.
Final number of references: 6306


In [19]:
combined_df.to_csv("TRIPOD_combined_references.csv", index=False)

In [26]:
with open("TRIPOD_PMIDs.txt", "w") as f:
    for name in sorted(int_names):
        f.write(f"{name}\n")